# Day 4 — Percentiles, IQR & Outlier Detection

**90-Day Data Science Roadmap — Phase 1: Statistics Foundations**

Today's goal: detect outliers in a numeric dataset using percentiles and the IQR rule, and understand why this method is more robust than standard deviation.

## 1. Build a sample dataset

Simulated house prices (in lakhs) — mostly normal houses, plus two mansions that should stick out as outliers.

In [1]:
import numpy as np

# Simulated house prices (in lakhs) - mostly normal houses, plus 2 mansions
house_prices = np.array([45, 50, 48, 52, 47, 49, 55, 51, 46, 53,
                           48, 50, 49, 52, 250, 47, 51, 48, 300, 50])

print(house_prices)

[ 45  50  48  52  47  49  55  51  46  53  48  50  49  52 250  47  51  48
 300  50]


## 2. Compute Q1, Q2 (median), Q3

Percentiles mark a **position** in sorted data, not a value's importance.

In [2]:
Q1 = np.percentile(house_prices, 25)
Q2 = np.percentile(house_prices, 50)  # this equals the median
Q3 = np.percentile(house_prices, 75)

print(f"Q1 (25th percentile): {Q1}")
print(f"Q2 (50th percentile / median): {Q2}")
print(f"Q3 (75th percentile): {Q3}")

Q1 (25th percentile): 47.75
Q2 (50th percentile / median): 50.0
Q3 (75th percentile): 52.25


Notice Q1 and Q3 sit tightly around 48-52, even though 250 and 300 are in the dataset. Percentiles are rank-based, so extreme values don't drag them anywhere — that's the robustness property.

## 3. Compute IQR and outlier bounds (Tukey's rule)

```
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
```

In [3]:
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"IQR: {IQR}")
print(f"Lower bound: {lower_bound}")
print(f"Upper bound: {upper_bound}")

IQR: 4.5
Lower bound: 40.99999999999999
Upper bound: 59.0


## 4. Detect outliers with boolean masking

In [4]:
# Boolean mask - True where price is outside the bounds
is_outlier = (house_prices < lower_bound) | (house_prices > upper_bound)

outliers = house_prices[is_outlier]
normal_values = house_prices[~is_outlier]  # ~ flips the mask

print(f"Outliers: {outliers}")
print(f"Normal values: {normal_values}")

Outliers: [250 300]
Normal values: [45 50 48 52 47 49 55 51 46 53 48 50 49 52 47 51 48 50]


Without ever manually scanning the numbers, Python correctly flagged exactly the two mansions (250 and 300).

## 5. Why IQR beats standard deviation here

- Q1/Q3 are computed from **rank/position** in sorted data — an outlier just becomes "the last item in line," it can't drag Q1/Q3 anywhere.
- Standard deviation is computed from actual values, **squared** — one extreme value inflates std dev itself, which can widen the "normal range" enough to swallow the outlier instead of flagging it (the masking effect).
- Practical rule: use IQR as the default for real-world, possibly skewed/messy data. Use std-dev / z-score methods only when data is already roughly normal and clean.

## 6. Common mistakes

| Mistake | Fix |
|---|---|
| Confusing percentile with percentage | Percentile = position in sorted data, never the raw value itself |
| Applying 1.5xIQR blindly on heavily skewed data | Check the distribution shape first (Day 5) - skewed data may need a log transform before outlier detection |
| Deleting every flagged outlier automatically | Investigate first - error vs. real extreme value decides the fix |

## 7. Today I learned

- Percentiles mark position in sorted data - extreme values can't move Q1/Q3 because rank doesn't care about magnitude.
- IQR is the safer default outlier detector for real-world data because std dev gets masked/inflated by the outliers themselves.
- A flagged outlier isn't automatically bad data - always investigate why before deciding to remove, keep, or transform.

**Next:** Day 5 - Distributions: Normal & Skewness